# 카트폴 게임 마스터하기

In [1]:
import gym  # OpenAI Gym 라이브러리 임포트
import random  # 무작위 행동을 위한 random 임포트
import math  # 수학 계산을 위한 math 임포트
import torch  # 파이토치 라이브러리 임포트
import torch.nn as nn  # 신경망 모듈 임포트
import torch.optim as optim  # 최적화 알고리즘 임포트
import torch.nn.functional as F  # 신경망 함수 임포트
from collections import deque  # 경험 재생용 큐(deque) 임포트
import matplotlib.pyplot as plt  # 시각화를 위한 matplotlib 임포트

### 하이퍼파라미터



In [2]:
# 하이퍼파라미터
EPISODES = 50    # 전체 에피소드 반복 횟수
EPS_START = 0.9  # 무작위 행동 확률의 초기값
EPS_END = 0.05   # 무작위 행동 확률의 최종값
EPS_DECAY = 200  # 무작위 행동 확률을 감소시키는 주기
GAMMA = 0.8      # 미래 보상에 대한 할인 계수
LR = 0.001       # 학습률
BATCH_SIZE = 64  # 한 번에 학습할 배치 크기

## DQN 에이전트

In [3]:
class DQNAgent:  # DQN 에이전트 클래스
    def __init__(self):  # 초기화
        self.model = nn.Sequential(  # 인공신경망 모델 정의
            nn.Linear(4, 256),  # 입력 상태(4차원), 은닉층(256)
            nn.ReLU(),  # 활성화 함수 ReLU
            nn.Linear(256, 2)  # 출력 행동(2차원)
        )
        self.optimizer = optim.Adam(self.model.parameters(), LR)  # Adam 최적화
        self.steps_done = 0  # 수행된 총 스텝 수
        self.memory = deque(maxlen=10000)  # 경험 저장소 생성

    def memorize(self, state, action, reward, next_state):  # 경험 저장 메서드
        self.memory.append((state,  # 현재 상태
                            action,  # 수행한 행동
                            torch.FloatTensor([reward]),  # 보상
                            torch.FloatTensor([next_state])))  # 다음 상태
    
    def act(self, state):  # 행동 선택 메서드
        eps_threshold = EPS_END + (EPS_START - EPS_END) * math.exp(-1. * self.steps_done / EPS_DECAY)  # 엡실론 계산
        self.steps_done += 1  # 스텝 증가
        if random.random() > eps_threshold:  # 확률에 따라 신경망 활용
            return self.model(state).data.max(1)[1].view(1, 1)  # 최대 Q값 행동 선택
        else:  # 확률에 따라 무작위 행동
            return torch.LongTensor([[random.randrange(2)]])  # 0 또는 1 무작위 선택
    
    def learn(self):  # 학습 메서드
        if len(self.memory) < BATCH_SIZE:  # 메모리가 배치 크기보다 작으면 리턴
            return
        batch = random.sample(self.memory, BATCH_SIZE)  # 무작위 배치 샘플링
        states, actions, rewards, next_states = zip(*batch)  # 데이터 분리

        states = torch.cat(states)  # 상태 텐서 결합
        actions = torch.cat(actions)  # 행동 텐서 결합
        rewards = torch.cat(rewards)  # 보상 텐서 결합
        next_states = torch.cat(next_states)  # 다음 상태 텐서 결합

        current_q = self.model(states).gather(1, actions)  # 현재 상태의 Q값
        max_next_q = self.model(next_states).detach().max(1)[0]  # 다음 상태의 최대 Q값
        expected_q = rewards + (GAMMA * max_next_q)  # 타겟 Q값 계산
        
        loss = F.mse_loss(current_q.squeeze(), expected_q)  # MSE 손실 계산
        self.optimizer.zero_grad()  # 기울기 초기화
        loss.backward()  # 역전파
        self.optimizer.step()  # 가중치 업데이트

## 학습 준비하기

`gym`을 이용하여 `CartPole-v0`환경을 준비하고 앞서 만들어둔 DQNAgent를 agent로 인스턴스화 합니다.

자, 이제 `agent` 객체를 이용하여 `CartPole-v0` 환경과 상호작용을 통해 게임을 배우도록 하겠습니다.

In [4]:
env = gym.make('CartPole-v0')  # 카트폴 환경 생성
agent = DQNAgent()  # 에이전트 인스턴스 생성
score_history = []  # 점수 기록 리스트

## 학습 시작

In [5]:
for e in range(1, EPISODES+1):  # 설정한 에피소드만큼 반복
    state = env.reset()  # 환경 초기화
    steps = 0  # 스텝 초기화
    while True:  # 에피소드 종료까지 반복
        env.render()  # 화면 출력
        state = torch.FloatTensor([state])  # 현재 상태를 텐서로 변환
        action = agent.act(state)  # 에이전트가 행동 선택
        next_state, reward, done, _ = env.step(action.item())  # 환경에 행동 전달

        # 게임이 끝났을 경우 마이너스 보상주기
        if done:  # 종료 상태인 경우
            reward = -1  # 패널티 부여

        agent.memorize(state, action, reward, next_state)  # 경험 저장
        agent.learn()  # 학습 수행

        state = next_state  # 상태 업데이트
        steps += 1  # 스텝 증가

        if done:  # 에피소드 종료 시
            print("에피소드:{0} 점수: {1}".format(e, steps))  # 결과 출력
            score_history.append(steps)  # 점수 저장
            break  # 반복 중단

In [7]:
plt.plot(score_history)  # 점수 그래프 그리기
plt.ylabel('score')  # y축 이름 설정
plt.show()  # 그래프 출력